## Homework: Multilingual Embedding-based Machine Translation (6 points)

**In this homework** **<font color='red'>YOU</font>** will make a machine translation system without using parallel corpora, alignment, attention, 100500-depth super-cool recurrent neural networks and all that kind of superstuff.

But even without parallel corpora, this system can be good enough (hopefully). 

For our system we choose two kindred Slavic languages: Ukrainian and Russian. 

### Feel the difference!

(_синій кіт_ vs. _синій кит_)

![blue_cat_blue_whale.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/blue_cat_blue_whale.png)

### Fragment of the Swadesh list for some Slavic languages

The Swadesh list is a lexicostatistical tool. It's named after American linguist Morris Swadesh and contains basic lexis. These lists are used to define subgroupings of languages and their relatedness.

So we can see some kind of word invariance for different Slavic languages.


| Russian         | Belorussian              | Ukrainian               | Polish             | Czech                         | Bulgarian            |
|-----------------|--------------------------|-------------------------|--------------------|-------------------------------|-----------------------|
| женщина         | жанчына, кабета, баба    | жінка                   | kobieta            | žena                          | жена                  |
| мужчина         | мужчына                  | чоловік, мужчина        | mężczyzna          | muž                           | мъж                   |
| человек         | чалавек                  | людина, чоловік         | człowiek           | člověk                        | човек                 |
| ребёнок, дитя   | дзіця, дзіцёнак, немаўля | дитина, дитя            | dziecko            | dítě                          | дете                  |
| жена            | жонка                    | дружина, жінка          | żona               | žena, manželka, choť          | съпруга, жена         |
| муж             | муж, гаспадар            | чоловiк, муж            | mąż                | muž, manžel, choť             | съпруг, мъж           |
| мать, мама      | маці, матка              | мати, матір, неня, мама | matka              | matka, máma, 'стар.' mateř    | майка                 |
| отец, тятя      | бацька, тата             | батько, тато, татусь    | ojciec             | otec                          | баща, татко           |
| много           | шмат, багата             | багато                  | wiele              | mnoho, hodně                  | много                 |
| несколько       | некалькі, колькі         | декілька, кілька        | kilka              | několik, pár, trocha          | няколко               |
| другой, иной    | іншы                     | інший                   | inny               | druhý, jiný                   | друг                  |
| зверь, животное | жывёла, звер, істота     | тварина, звір           | zwierzę            | zvíře                         | животно               |
| рыба            | рыба                     | риба                    | ryba               | ryba                          | риба                  |
| птица           | птушка                   | птах, птиця             | ptak               | pták                          | птица                 |
| собака, пёс     | сабака                   | собака, пес             | pies               | pes                           | куче, пес             |
| вошь            | вош                      | воша                    | wesz               | veš                           | въшка                 |
| змея, гад       | змяя                     | змія, гад               | wąż                | had                           | змия                  |
| червь, червяк   | чарвяк                   | хробак, черв'як         | robak              | červ                          | червей                |
| дерево          | дрэва                    | дерево                  | drzewo             | strom, dřevo                  | дърво                 |
| лес             | лес                      | ліс                     | las                | les                           | гора, лес             |
| палка           | кій, палка               | палиця                  | patyk, pręt, pałka | hůl, klacek, prut, kůl, pálka | палка, пръчка, бастун |

But the context distribution of these languages demonstrates even more invariance. And we can use this fact for our purposes.

## Data

In [1]:
import gensim
import numpy as np
from gensim.models import KeyedVectors

Download embeddings here:
* [cc.uk.300.vec.zip](https://yadi.sk/d/9CAeNsJiInoyUA)
* [cc.ru.300.vec.zip](https://yadi.sk/d/3yG0-M4M8fypeQ)

Load embeddings for Ukrainian and Russian.

In [2]:
uk_emb = KeyedVectors.load_word2vec_format("cc.uk.300.vec")

In [3]:
ru_emb = KeyedVectors.load_word2vec_format("cc.ru.300.vec")

In [4]:
ru_emb.most_similar([ru_emb["апрель"]], topn=10)

[('апрель', 1.0),
 ('март', 0.9489257335662842),
 ('февраль', 0.9302883148193359),
 ('ноябрь', 0.9292629957199097),
 ('октябрь', 0.9264576435089111),
 ('сентябрь', 0.9242205023765564),
 ('декабрь', 0.9030115008354187),
 ('июнь', 0.8981205224990845),
 ('январь', 0.8968256115913391),
 ('август', 0.8729088306427002)]

In [5]:
uk_emb.most_similar([uk_emb["серпень"]])

[('серпень', 0.9999998807907104),
 ('липень', 0.9096441268920898),
 ('вересень', 0.9016969203948975),
 ('червень', 0.8992518782615662),
 ('жовтень', 0.8810408115386963),
 ('листопад', 0.8787633180618286),
 ('квітень', 0.8592804670333862),
 ('грудень', 0.8586863279342651),
 ('травень', 0.840811014175415),
 ('лютий', 0.8256431221961975)]

In [6]:
ru_emb.most_similar([uk_emb["червень"]], topn=10)

[('deteydlya', 0.2688758075237274),
 ('офор', 0.2543112635612488),
 ('иболее', 0.25083014369010925),
 ('функциональ', 0.23748886585235596),
 ('Недопустимость', 0.2372935265302658),
 ('Запорожцев', 0.23446659743785858),
 ('настоль', 0.22794149816036224),
 ('СЕКЦИЯ', 0.22560444474220276),
 ('коробкаЧехол', 0.22559276223182678),
 ('туристаФорумПоиск', 0.22551143169403076)]

Load small dictionaries for corresponding word pairs as training set and test set.

In [7]:
def load_word_pairs(filename):
    uk_ru_pairs = []
    uk_vectors = []
    ru_vectors = []
    with open(filename, "r", encoding="utf-8") as inpf:
        for line in inpf:
            uk, ru = line.rstrip().split("\t")
            if uk not in uk_emb or ru not in ru_emb:
                continue
            uk_ru_pairs.append((uk, ru))
            uk_vectors.append(uk_emb[uk])
            ru_vectors.append(ru_emb[ru])
    return uk_ru_pairs, np.array(uk_vectors), np.array(ru_vectors)

In [8]:
uk_ru_train, X_train, Y_train = load_word_pairs("ukr_rus.train.txt")

In [9]:
uk_ru_test, X_test, Y_test = load_word_pairs("ukr_rus.test.txt")

## Embedding space mapping

Let $x_i \in \mathrm{R}^d$ be the distributed representation of word $i$ in the source language, and $y_i \in \mathrm{R}^d$ is the vector representation of its translation. Our purpose is to learn such a linear transform $W$ that minimizes the Euclidean distance between $Wx_i$ and $y_i$ for some subset of word embeddings. Thus we can formulate the so-called Procrustes problem:

$$W^*= \arg\min_W \sum_{i=1}^n||Wx_i - y_i||_2$$
or
$$W^*= \arg\min_W ||WX - Y||_F$$

where $||*||_F$ is the Frobenius norm.

In Greek mythology, Procrustes or "the stretcher" was a rogue smith and bandit from Attica who attacked people by stretching them or cutting off their legs, so as to force them to fit the size of an iron bed. We do the same bad things with the source embedding space. Our Procrustean bed is the target embedding space.

![embedding_mapping.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/embedding_mapping.png)

![procrustes.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/procrustes.png)

But wait... $W^*= \arg\min_W \sum_{i=1}^n||Wx_i - y_i||_2$ looks like simple multiple linear regression (without intercept fit). So let's code.

In [10]:
from sklearn.linear_model import LinearRegression

# YOUR CODE HERE
mapping = LinearRegression(fit_intercept=False).fit(X_train, Y_train)

Let's take a look at neighbors of the vector of word _"серпень"_ (_"август"_ in Russian) after linear transform.

In [11]:
august = mapping.predict(uk_emb["серпень"].reshape(1, -1))
ru_emb.most_similar(august)

[('апрель', 0.8541285991668701),
 ('июнь', 0.8411202430725098),
 ('март', 0.839699387550354),
 ('сентябрь', 0.8359869718551636),
 ('февраль', 0.832929790019989),
 ('октябрь', 0.8311846852302551),
 ('ноябрь', 0.8278924226760864),
 ('июль', 0.8234530091285706),
 ('август', 0.8120501637458801),
 ('декабрь', 0.8039005398750305)]

We can see that the neighborhood of this embedding consists of different months, but the right variant is in ninth place.

As a quality measure we will use precision top-1, top-5 and top-10 (for each transformed Ukrainian embedding we count how many correct target pairs are found in the top N nearest neighbors in Russian embedding space).

In [12]:
def precision(pairs, mapped_vectors, topn=1):
    """
    :args:
        pairs = list of correct word pairs [(uk_word_0, ru_word_0), ...]
        mapped_vectors = list of embeddings after mapping from source embedding space to destination embedding space
        topn = the number of nearest neighbors in destination embedding space to choose from
    :returns:
        precision_val, float number, total number of words for which we can find correct translation in top K.
    """
    assert len(pairs) == len(mapped_vectors)
    num_matches = 0
    for i, (_, ru) in enumerate(pairs):
        nearest_neighbors = ru_emb.similar_by_vector(mapped_vectors[i], topn=topn)
        if ru in [neighbor for neighbor, _ in nearest_neighbors]:
            num_matches += 1    
        # pass
    precision_val = num_matches / len(pairs)
    return precision_val


In [13]:
pairs = [("серпень", "август")]
mapped_vectors = august
topn=9  
num_matches = 0
for i, (_, ru) in enumerate(pairs):
        nearest_neighbors = ru_emb.similar_by_vector(mapped_vectors[i], topn=topn)
        if ru in [neighbor for neighbor, _ in nearest_neighbors]:
            num_matches += 1   

In [14]:
nearest_neighbors

[('апрель', 0.8541285991668701),
 ('июнь', 0.8411202430725098),
 ('март', 0.839699387550354),
 ('сентябрь', 0.8359869718551636),
 ('февраль', 0.832929790019989),
 ('октябрь', 0.8311846852302551),
 ('ноябрь', 0.8278924226760864),
 ('июль', 0.8234530091285706),
 ('август', 0.8120501637458801)]

In [15]:
assert precision([("серпень", "август")], august, topn=5) == 0.0
assert precision([("серпень", "август")], august, topn=9) == 1.0
assert precision([("серпень", "август")], august, topn=10) == 1.0

In [16]:
assert precision(uk_ru_test, X_test) == 0.0
assert precision(uk_ru_test, Y_test) == 1.0

In [17]:
precision_top1 = precision(uk_ru_test, mapping.predict(X_test), 1)
precision_top5 = precision(uk_ru_test, mapping.predict(X_test), 5)

assert precision_top1 >= 0.635
assert precision_top5 >= 0.813

W_true:
[[ 6.123234e-17 -1.000000e+00]
 [ 1.000000e+00  6.123234e-17]]

W_hat:
[[ 6.22328532e-19 -1.00000000e+00]
 [ 1.00000000e+00  2.11898069e-16]]

Frobenius error:
7.45915314199903e-16


In [32]:
import numpy as np

def orthogonal_procrustes(X, Y):
    U, S, Vt = np.linalg.svd(X.T @ Y)
    W = U @ Vt
    return W

X = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

Y = np.array([
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])


M = np.array([[3., 1.],
              [1., 2.]])

In [29]:
np.linalg.svd(X.T @ Y)

SVDResult(U=array([[-0.70710678, -0.70710678],
       [-0.70710678,  0.70710678]]), S=array([3., 1.]), Vh=array([[-0.70710678,  0.70710678],
       [ 0.70710678,  0.70710678]]))

In [30]:
print("U =\n", U)  # матрица (2x2), ортогональная
print("S =", S)  # вектор [3.73, 1.27], сингулярные числа
print("Vt =\n", Vt)  # матрица (2x2), ортогональная


U =
 [[-0.70710678 -0.70710678]
 [-0.70710678  0.70710678]]
S = [3. 1.]
Vt =
 [[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


In [33]:
# Проверяем восстановление
print("\nU @ diag(S) @ Vt =\n", U @ np.diag(S) @ Vt)
print("M =\n", M)



U @ diag(S) @ Vt =
 [[ 1. -2.]
 [ 2. -1.]]
M =
 [[3. 1.]
 [1. 2.]]


In [34]:

# Оптимальный поворот (Sigma отброшена)
W = U @ Vt
print("\nW = U @ Vt =\n", W)
print("Ортогональность: W.T @ W =\n", W.T @ W)  # должна быть I


W = U @ Vt =
 [[ 6.22328532e-19 -1.00000000e+00]
 [ 1.00000000e+00  2.11898069e-16]]
Ортогональность: W.T @ W =
 [[1.0000000e+00 2.1127574e-16]
 [2.1127574e-16 1.0000000e+00]]


In [23]:
theta = np.deg2rad(90)
theta

np.float64(1.5707963267948966)

In [24]:

W_true = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)],
])


In [25]:
W_true

array([[ 6.123234e-17, -1.000000e+00],
       [ 1.000000e+00,  6.123234e-17]])

In [26]:
Y = X @ W_true


In [27]:
W_hat = orthogonal_procrustes(X, Y)

print("W_true:")
print(W_true)

print("\nW_hat:")
print(W_hat)

print("\nFrobenius error:")
print(np.linalg.norm(X @ W_hat - Y, ord="fro"))

W_true:
[[ 6.123234e-17 -1.000000e+00]
 [ 1.000000e+00  6.123234e-17]]

W_hat:
[[ 6.22328532e-19 -1.00000000e+00]
 [ 1.00000000e+00  2.11898069e-16]]

Frobenius error:
7.45915314199903e-16


## Making it better (orthogonal Procrustean problem)

It can be shown (see [original paper](https://arxiv.org/pdf/1710.04087)) that a self-consistent linear mapping between semantic spaces should be orthogonal. 
We can restrict the transform $W$ to be orthogonal. Then we will solve the next problem:

$$W^*= \arg\min_W ||WX - Y||_F \text{, where: } W^TW = I$$

$$I \text{ - identity matrix}$$

Instead of making yet another regression problem, we can find the optimal orthogonal transformation using singular value decomposition. It turns out that the optimal transformation $W^*$ can be expressed via SVD components:
$$X^TY=U\Sigma V^T\text{, singular value decomposition}$$
$$W^*=UV^T$$

In [35]:
def learn_transform(X_train, Y_train):
    """ 
    :returns: W* : float matrix[emb_dim x emb_dim] as defined in formulae above
    """
    U, S, Vt = np.linalg.svd(X_train.T @ Y_train)
    W = U @ Vt
    return W

In [36]:
W = learn_transform(X_train, Y_train)

In [37]:
ru_emb.most_similar([np.matmul(uk_emb["серпень"], W)])

[('апрель', 0.8237908482551575),
 ('сентябрь', 0.8049712181091309),
 ('март', 0.802565336227417),
 ('июнь', 0.8021841645240784),
 ('октябрь', 0.8001735806465149),
 ('ноябрь', 0.7934483289718628),
 ('февраль', 0.7914120554924011),
 ('июль', 0.7908107042312622),
 ('август', 0.7891014814376831),
 ('декабрь', 0.7686371803283691)]

In [38]:
assert precision(uk_ru_test, np.matmul(X_test, W)) >= 0.653
assert precision(uk_ru_test, np.matmul(X_test, W), 5) >= 0.824

## UK-RU Translator

Now we are ready to make a simple word-based translator: for each word in the source language in shared embedding space we find the nearest in the target language.


In [ ]:
with open("fairy_tale.txt", "r", encoding="utf-8") as inpf:
    uk_sentences = [line.rstrip().lower() for line in inpf]

In [ ]:
def translate(sentence):
    """
    :args:
        sentence - sentence in Ukrainian (str)
    :returns:
        translation - sentence in Russian (str)

    * find Ukrainian embedding for each word in sentence
    * transform Ukrainian embedding vector
    * find nearest Russian word and replace
    """
    # YOUR CODE HERE
    pass

In [ ]:
assert translate(".") == "."
assert translate("1 , 3") == "1 , 3"
assert translate("кіт зловив мишу") == "кот поймал мышку"

In [ ]:
for sentence in uk_sentences:
    print("src: {}\ndst: {}\n".format(sentence, translate(sentence)))

Not so bad, right? We can easily improve translation using a language model and not one but several nearest neighbors in shared embedding space. But that's for next time.

## Would you like to learn more?

### Articles:
* [Exploiting Similarities among Languages for Machine Translation](https://arxiv.org/pdf/1309.4168)  - entry point for multilingual embedding studies by Tomas Mikolov (the author of W2V)
* [Offline bilingual word vectors, orthogonal transformations and the inverted softmax](https://arxiv.org/pdf/1702.03859) - orthogonal transform for unsupervised MT
* [Word Translation Without Parallel Data](https://arxiv.org/pdf/1710.04087)
* [Loss in Translation: Learning Bilingual Word Mapping with a Retrieval Criterion](https://arxiv.org/pdf/1804.07745)
* [Unsupervised Alignment of Embeddings with Wasserstein Procrustes](https://arxiv.org/pdf/1805.11222)

### Repos (with ready-to-use multilingual embeddings):
* https://github.com/facebookresearch/MUSE

* https://github.com/Babylonpartners/fastText_multilingual -